# Employetics — 40s Complete Motion Graphics Video Ad (Google Colab GPU)

Render the full **40-second, 1080p / 4K @ 60fps** SaaS motion-graphics commercial for **Employetics** with:
- **Remotion** (React + TypeScript + SVG + Spring Physics)
- **Kokoro-82M GPU Neural Voiceover** (studio commercial quality, 100% offline on GPU, with automatic Edge-TTS / gTTS fallback)
- **Custom Procedural Modern SaaS Soundtrack + Synchronized UI SFX** with automatic voiceover sidechain ducking
- **NVIDIA GPU Hardware Encoding (`h264_nvenc`)** + Chrome GPU acceleration

> **Before running:** Go to **Runtime → Change runtime type** and select **T4 GPU** (or **L4 / A100 GPU**), then click **Runtime → Run all**.

In [ ]:
#@title 1. Configure Your Render Settings
GITHUB_REPO_URL = "https://github.com/abdullahmehmood1/EmployeticsAD.git" #@param {type:"string"}
BRANCH = "main" #@param {type:"string"}
TTS_ENGINE = "kokoro" #@param ["kokoro", "auto", "edge", "gtts"]
KOKORO_VOICE = "am_michael" #@param ["am_michael", "am_fenrir", "am_adam", "af_heart", "af_bella", "bm_george"]
VOICE_SPEED = 1.02 #@param {type:"slider", min:0.85, max:1.20, step:0.01}
MUSIC_VOLUME = 0.72 #@param {type:"slider", min:0.0, max:1.0, step:0.02}
BURN_CAPTIONS = False #@param {type:"boolean"}
RESOLUTION = "1080p (1920x1080 @ 60fps)" #@param ["1080p (1920x1080 @ 60fps)", "4K (3840x2160 @ 60fps)"]
SAVE_TO_GOOGLE_DRIVE = False #@param {type:"boolean"}

SCALE = 2.0 if RESOLUTION.startswith("4K") else 1.0
OUTPUT_FILENAME = "employetics-ad-4k60.mp4" if SCALE >= 2.0 else "employetics-ad-1080p60.mp4"
print(f"Configured: {RESOLUTION} | Engine={TTS_ENGINE} | Voice={KOKORO_VOICE} | Captions={BURN_CAPTIONS}")


In [ ]:
#@title 2. Verify GPU & Install System / Node 22 / Chrome / Kokoro Dependencies
import os, shutil, subprocess

print("=== Checking NVIDIA GPU ===")
gpu_check = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
if gpu_check.returncode == 0:
    print("GPU ACTIVE:", gpu_check.stdout.strip())
else:
    print("WARNING: No NVIDIA GPU detected. Switch Runtime -> Change runtime type -> T4 GPU for fastest rendering.")

print("\n=== Installing system packages (Chrome GPU libs, espeak-ng, ffmpeg) ===")
!apt-get -qq update
!apt-get -qq install -y ffmpeg espeak-ng libnss3 libatk1.0-0 libatk-bridge2.0-0 libcups2 libdrm2 libxkbcommon0 libxcomposite1 libxdamage1 libxfixes3 libxrandr2 libgbm1 libasound2 libpango-1.0-0 libcairo2 libegl1 libgles2

print("\n=== Installing Node.js 22 ===")
node_ver = subprocess.run(["node", "-v"], capture_output=True, text=True)
if node_ver.returncode != 0 or not node_ver.stdout.strip().startswith(("v20", "v22", "v24")):
    !curl -fsSL https://deb.nodesource.com/setup_22.x | bash - >/dev/null 2>&1
    !apt-get -qq install -y nodejs
!node -v && npm -v

print("\n=== Cloning / Updating Project Repository ===")
PROJECT_DIR = "/content/employeticsAD"
if os.path.exists(os.path.join(PROJECT_DIR, ".git")):
    !git -C {PROJECT_DIR} fetch origin && git -C {PROJECT_DIR} checkout {BRANCH} && git -C {PROJECT_DIR} pull origin {BRANCH}
else:
    if os.path.exists(PROJECT_DIR):
        shutil.rmtree(PROJECT_DIR)
    !git clone --branch {BRANCH} {GITHUB_REPO_URL} {PROJECT_DIR}

%cd /content/employeticsAD

print("\n=== Installing Python Audio & Neural TTS Dependencies ===")
!pip install -q -r requirements.txt

print("\n=== Installing Node / Remotion Packages & Headless Chrome ===")
!npm install --no-audit --no-fund
!npx remotion browser ensure
print("\nSetup complete!")


In [ ]:
#@title 3. Generate Studio Neural Voiceover + Soundtrack + UI SFX + Captions
%cd /content/employeticsAD
from IPython.display import Audio, display

!python scripts/generate_audio.py \
  --engine {TTS_ENGINE} \
  --voice {KOKORO_VOICE} \
  --speed {VOICE_SPEED} \
  --music-volume {MUSIC_VOLUME}

print("\nPreview your mastered 40-second audio track below:")
display(Audio("public/audio/master_audio.wav", autoplay=False))


In [ ]:
#@title 4. Render 40-Second Motion Graphics Commercial (GPU Accelerated)
%cd /content/employeticsAD

caption_flag = "--captions" if BURN_CAPTIONS else ""
out_path = f"out/{OUTPUT_FILENAME}"

!python scripts/render_video.py \
  --output {out_path} \
  --scale {SCALE} \
  --crf 16 \
  {caption_flag}


In [ ]:
#@title 5. Preview Rendered Video Inline
from IPython.display import Video, display
import os

video_path = f"/content/employeticsAD/out/{OUTPUT_FILENAME}"
assert os.path.exists(video_path), f"Video not found at {video_path}"
print(f"Rendered video size: {os.path.getsize(video_path) / (1024*1024):.2f} MB")
display(Video(video_path, embed=True, width=960))


In [ ]:
#@title 6. Download MP4 + SRT Captions (and Optional Google Drive Backup)
import os, shutil
from google.colab import files

video_path = f"/content/employeticsAD/out/{OUTPUT_FILENAME}"
srt_path = "/content/employeticsAD/public/audio/captions.srt"

if SAVE_TO_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    dest_dir = "/content/drive/MyDrive/EmployeticsAd"
    os.makedirs(dest_dir, exist_ok=True)
    shutil.copy2(video_path, os.path.join(dest_dir, OUTPUT_FILENAME))
    if os.path.exists(srt_path):
        shutil.copy2(srt_path, os.path.join(dest_dir, "captions.srt"))
    print(f"Saved copies to Google Drive: {dest_dir}")

files.download(video_path)
if os.path.exists(srt_path):
    files.download(srt_path)
